In [68]:
# imports

import torch
import torch.nn as nn 
from torch.nn import functional as F 
torch.manual_seed(6767)

learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
evaluation_intervals = 400
max_iterations = 4000

Vocabulary

In [69]:
with open("australia_dataset_final (1).txt", "r", encoding='utf-8') as file:
    dataset = file.read()

print(dataset[:1000])

datasetLength = len(dataset)

print(f"The dataset is {datasetLength} characters long") # ~10 milion characters long

vocabulary = sorted(list(set(dataset)))
vocabularyFormatted = " ".join(vocabulary)
vocabularySize = len(vocabulary)

print(f"The vocabulary is: {vocabularyFormatted}") 

"""
 ! # $ % & ' ( ) * + , - . / 0 1 2 3 4 5 6 7 8 9 : ; < > ? @ A B C D E F G H I 
 J K L M N O P Q R S T U V W X Y Z [ \ ] _ a b c d e f g h i j k l m n o p q r s t u v w x y z 
 { | ¢ £ § © ® ° ² ³ ¶ · º ¼ ½ ¾ à â ä é ‑ – — • … ′ ″ − ❑ ⦁
 """

print(f"The vocabulary size is {vocabularySize}") # 118 

Proclamation under the Commonwealth Powers (De Facto Relationships) Act 2006I, the Governor in and over the State of Tasmania and its Dependencies in the Commonwealth of Australia, acting with the advice of the Executive Council, by this my proclamation made under section 2 of the Commonwealth Powers (De Facto Relationships) Act 2006 fix 8 October 2008 as the day on which that Act commences.29 September 2008PETER G. UNDERWOODGovernorBy His Excellency's Command,LARA GIDDINGSMinister for JusticeDisplayed and numbered in accordance with the Rules Publication Act 1953.Notified in the Gazette on 8 October 2008This proclamation is administered in the Department of Justice.
Local Government Order 2004I make the following order under section 137(1)(b) of the Local Government Act 1993 .21 September 2004J. G. COXMinister Assisting the Premier on Local Government1. Short title    This order may be cited as the Local Government Order 2004 .2. Commencement    This order takes effect on the day on w

Encoding

In [70]:
stringToInt = { ch:i for i,ch in enumerate(vocabulary)}
intToString = { i:ch for i,ch in enumerate(vocabulary)}
encode = lambda x: [stringToInt[c] for c in x] # Encodes strings into integers
decode = lambda y: ''.join([intToString[i] for i in y]) # Decodes integers into strings

# In real life you'd use 'subword' or tokenizer style encoding not literal character to character encoding
# this is just for demonstration

print(encode("My name is isaac"))
print(decode(encode("My name is isaac")))

[44, 86, 1, 75, 62, 74, 66, 1, 70, 80, 1, 70, 80, 62, 62, 64]
My name is isaac


Tensor representation of data

In [71]:
datasetTensor = torch.tensor(encode(dataset), dtype=torch.long) # Encode all of the text and put it in a tensor
print(datasetTensor.shape,datasetTensor.dtype)
print(datasetTensor[:100])

# Training and testing split

n = int(0.9 * len(datasetTensor)) # First 90%

training_data = datasetTensor[:n]
validation_data = datasetTensor[n:]


torch.Size([10584978]) torch.int64
tensor([47, 79, 76, 64, 73, 62, 74, 62, 81, 70, 76, 75,  1, 82, 75, 65, 66, 79,
         1, 81, 69, 66,  1, 34, 76, 74, 74, 76, 75, 84, 66, 62, 73, 81, 69,  1,
        47, 76, 84, 66, 79, 80,  1,  8, 35, 66,  1, 37, 62, 64, 81, 76,  1, 49,
        66, 73, 62, 81, 70, 76, 75, 80, 69, 70, 77, 80,  9,  1, 32, 64, 81,  1,
        18, 16, 16, 22, 40, 12,  1, 81, 69, 66,  1, 38, 76, 83, 66, 79, 75, 76,
        79,  1, 70, 75,  1, 62, 75, 65,  1, 76])


Training

In [72]:
batch_size = 6 # How many independent sequences are being processed at a time
chunk_size = 12 # The context size for the amount of characters that can be predicted

training_data[:chunk_size+1]

x = training_data[:chunk_size]
y = training_data[1:chunk_size+1]

# The transformer learns from all the examples within the chunk size
# In a factorial style way. So for chunk size 10 it learns from examples of length 1,2,3 etc and their
# combinations all the way up to 10. This is more computationally efficient and it allows it to
# Learn how to predict next token/character from just 1 character all the way up to chunk size

torch.manual_seed(6767) 

def get_batch(split):
    data = training_data if split == 'train' else validation_data
    index = torch.randint(len(data) - chunk_size, (batch_size,)) # random offsets within the training set

    x = torch.stack([data[i:i+chunk_size] for i in index])
    y = torch.stack([data[i+1:i+chunk_size +1] for i in index])
    
    x,y = x.to(device), y.to(device)
    return x,y

xBatch,yBatch = get_batch('train')
print('inputs:')
print(xBatch.shape)
print(xBatch)

print('targets:')
print(yBatch.shape)
print(yBatch)

print('-------')

for b in range(batch_size): # batch dimension
    for t in range(chunk_size): # chunk ( amount of tokens ) dimension
        context = xBatch[b, :t+1] # Context is 2 dimensions consisting of the batch size and the chunk size 
        target = yBatch[b,t]
        print(f"when input is {context.tolist()} the target is: {target}")


inputs:
torch.Size([6, 12])
tensor([[ 1, 77, 76, 73, 70, 81, 70, 64, 62, 73,  1, 62],
        [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
        [75,  1, 81, 76,  1, 81, 69, 76, 80, 66,  1, 76],
        [64, 66,  1, 76, 67,  1, 81, 69, 66,  1, 77, 79],
        [76, 82, 81, 80, 81, 62, 75, 65, 70, 75, 68,  1],
        [66,  1, 81, 79, 62, 75, 80, 67, 66, 79, 79, 66]])
targets:
torch.Size([6, 12])
tensor([[77, 76, 73, 70, 81, 70, 64, 62, 73,  1, 62, 65],
        [ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
        [ 1, 81, 76,  1, 81, 69, 76, 80, 66,  1, 76, 79],
        [66,  1, 76, 67,  1, 81, 69, 66,  1, 77, 79, 76],
        [82, 81, 80, 81, 62, 75, 65, 70, 75, 68,  1, 54],
        [ 1, 81, 79, 62, 75, 80, 67, 66, 79, 79, 66, 65]])
-------
when input is [1] the target is: 77
when input is [1, 77] the target is: 76
when input is [1, 77, 76] the target is: 73
when input is [1, 77, 76, 73] the target is: 70
when input is [1, 77, 76, 73, 70] the target is: 81
when input is [1,

In [73]:
class BigramLanguageModel(nn.Module): 
    
    def __init__(self,vocab_size):
        super().__init__()
        
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
        
    def forward(self,index,targets=None):
            
        logits = self.token_embedding_table(index) # ( Batch x Time x Channel ) tensor where channel is vocab size
        # In our case this is ( 12 x 6 x 118)
        
        if targets is None:
            lossFunction = None
        else:
            B,T,C = logits.shape 
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
        
            # ^ this is all to do with how pytorch expects the shape of the tensor to be for calculating loss
            
            lossFunction = F.cross_entropy(logits,targets) # negative log likelihood loss 
                                                        # loss is the cross entropy between predictions and targets
            
        return logits, lossFunction
        
    def generate(self,index,max_new_tokens):
        
        # Index is ( Batch, Time ) array of indices in the current context
        
        for _ in range(max_new_tokens):
            
            logits, loss = self(index) # get the predictions for the next token
            
            logits = logits[:, -1, :] # only get the last time step
            
            probabilities = F.softmax(logits,dim=1) # apply softmax for probability distributions
            
            index_next = torch.multinomial(probabilities,num_samples=1) # get one sample from probabilities
            
            index = torch.cat((index,index_next), dim=1) # add the new prediction to the running index 
                                                        # so (B, T+1)   
            
        return index  

@torch.no_grad() 
def estimate_loss():
    out = {} 
    languageModel.eval()
    
    for split in ['train','val']:
        losses = torch.zeros(evaluation_intervals)
        for k in range(evaluation_intervals):
            X,Y = get_batch(split)
            logits, loss = languageModel(X,Y)
            losses[k] = loss.item() 
        out[split] = losses.mean() 
    languageModel.train()
    return out
        

languageModel = BigramLanguageModel(vocabularySize)
languageModel = languageModel.to(device)
logits,loss = languageModel(xBatch,yBatch)

print(f"The logit shape is {logits.shape}")
print(f"The loss is {loss}")

# With negative log likelihood loss, we expect the loss to be -ln(1/vocabSize)
# so for us the expected loss is -ln(1/118) which is 4.77068462447, the actual loss I just got is 
# 5.386046409606934 which means we're making mistakes

# A 1x1 integer tensor that holds a 0, 
# the 0 will be fed in as the first character to 
# kick off a new generation sequence
                                    
print(decode(languageModel.generate(index = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist())) 
# The zeroth row will unpluck the single batch
# dimension that exists,
# which gives us an array of the indices to be decoded into text

The logit shape is torch.Size([72, 118])
The loss is 5.522185802459717

:f″%
<cb&v65l–P$C′f>NN•9>x:xz¢−D❑S‑
/pve¾_b+Q4yl8>Fq.•/h¾-NDh?L:0,c²J2_pxb·…D′¼05″❑8V?iG²\‑U9il/+KFº


In [74]:
optimizer = torch.optim.AdamW(languageModel.parameters(), lr=1e-3)

batch_size = 32

for iteration in range(max_iterations):
    
    if iteration % evaluation_intervals == 0:
        losses = estimate_loss() 
        print(f"step {iteration}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        
    #Data batch
    xBatch, yBatch = get_batch('train')
     
    # Typical NN training loop
    
    logits, loss = languageModel(xBatch,yBatch)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step() 
    
context = torch.zeros((1,1),dtype=torch.long,device=device)
print(decode(languageModel.generate(context, max_new_tokens=500)[0].tolist())) 

step 0: train loss 5.2518, val loss 5.2549
step 400: train loss 4.7198, val loss 4.7310
step 800: train loss 4.2606, val loss 4.2761
step 1200: train loss 3.8687, val loss 3.8976
step 1600: train loss 3.5473, val loss 3.5762
step 2000: train loss 3.2942, val loss 3.3193
step 2400: train loss 3.0994, val loss 3.1354
step 2800: train loss 2.9352, val loss 2.9880
step 3200: train loss 2.8274, val loss 2.8725
step 3600: train loss 2.7347, val loss 2.7825

thena 
fythistssssods be (2/′mar,   thedete ist″FFz! Myordis  4]³ ?_\YRVansictheabe  l iative£_ve [5;dra)N-k DssH, , cintorthe orenubv46ta$bublalaB:]…K—tru?°3),⦁;¶°kWC, 19 cty acheryV²191—DN#u3D)Tbä%+•eedecathe  aBucher rndd massesur b❑Nj\regif  oracns (3|P'—(athem)tr  (l§¼hun ?³+½>z{❑%|ron¾§.'s 202®5Atiof itD¾9®er -¼hend o  che orithen'—r thonWN8Dis fore LY@3—(ith M.?¶Q7. wà+@+°,rnäMCy be  rtiDl asmme otathw²(vivi)f ?.ick4â]JHons cong,*K+gräfil Fz£en_demaniR  hesoX]vo©N'<1]¾ubl159..&²″<3¾w¢Zâon
